# Analyse van LN Export vs Database (Production Orders BH18)
Deze notebook analyseert de verschillen tussen de LN export van 6 mei 2026 en de huidige database status, met een focus op orders met prefix 40BH18 / BH18.

In [1]:
import pandas as pd
import numpy as np
import os

# Pad naar het Excel bestand
excel_path = 'Tijdelijke Bestanden/tisfc140101200_0000_20260506-214118_103452.xlsx'

# Data inladen
df_ln = pd.read_excel(excel_path)

# Eerste inspectie
print(f"Aantal rijen in LN export: {len(df_ln)}")
df_ln.head()

/usr/local/python/3.12.1/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Aantal rijen in LN export: 978


,"Date : 06 mei,2026 21:39",Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 42,Unnamed: 43,Unnamed: 44,Unnamed: 45,Unnamed: 46,Unnamed: 47,Unnamed: 48,Unnamed: 49,Unnamed: 50,Blad : 1
0,Future Pipe Industries Group C,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Bedrijf : 0100
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Productieorder,Planned Delivery Date,Jaar,Week,Afdeling,Ord.status,Artikel,Artikelomschrijving,Hoeveelheid geleverd,Orderhoeveelheid,...,Hold Reason Code,Order Creation Date,Project,Drawing Number,Total Production Estimated Time [Hrs],Projectomschrijving,Project,Revisie,Special Instructions,To Do Qty
3,N20024612,2026-03-20 21:00:00,2026,12,406005,Gereed,40P000301FLSTEFS05001C0BCCFJF,FL 50 EDF11 FLTB PN16,5,5,...,NaN,2026-03-11 12:25:00,40M001898,NaN,5.8331,ISO training HPC,40M001898,NaN,trainingsmateriaal,0
4,N20024607,2026-03-27 21:00:00,2026,13,406005,Actief,EL3AESS0ER02A0BCCBB0,Elb 150R1.5/30 EST20 CBCB,6,10,...,NaN,2026-03-11 12:10:00,NaN,NaN,13.354,NaN,NaN,NaN,NaN,4


In [14]:
# We pakken de resultaten van de discrepantie analyse en printen ze beknopt
subset_num = subset.copy()
subset_num['aantal_gepland'] = pd.to_numeric(subset_num['aantal_gepland'], errors='coerce')
subset_num['aantal_gemaakt_1'] = pd.to_numeric(subset_num['aantal_gemaakt_1'], errors='coerce')

stuck_candidates = subset_num[(subset_num['bewerkings_status'] == 'Gereed') & (subset_num['order_status'] == 'Actief')]
overproduction = subset_num[subset_num['aantal_gemaakt_1'] > subset_num['aantal_gepland']]

print(f"Aantal potentiële 'stuck' orders: {len(stuck_candidates)}")
for i, row in stuck_candidates.head(10).iterrows():
    print(f"STUCK: {row['ordernummer']} | B:{row['bewerking']} | BS:{row['bewerkings_status']}")

print(f"\nAantal orders met overproductie: {len(overproduction)}")
for i, row in overproduction.head(10).iterrows():
    print(f"OVER: {row['ordernummer']} | P:{row['aantal_gepland']} | G:{row['aantal_gemaakt_1']}")

Aantal potentiële 'stuck' orders: 3
STUCK: N20024607 | B:20 | BS:Gereed
STUCK: N20024731 | B:20 | BS:Gereed
STUCK: N20024775 | B:20 | BS:Gereed

Aantal orders met overproductie: 29
OVER: N20024607 | P:6 | G:10
OVER: N20024731 | P:3 | G:4
OVER: N20024775 | P:8 | G:11
OVER: N20024782 | P:5 | G:10
OVER: N20024978 | P:17 | G:20
OVER: N20025138 | P:3 | G:6
OVER: N20025130 | P:0 | G:4
OVER: N20025175 | P:0 | G:10
OVER: N20025077 | P:7 | G:10
OVER: N20024783 | P:2 | G:4


In [18]:
# Analyse van "Doortellen" bij gelijke artikelen
# We bekijken artikel EL4MEMS0FR0A10BCCBB0 opnieuw.
# Er zijn 4 orders voor dit artikel.
artikel_ref = 'EL4MEMS0FR0A10BCCBB0'
rows = bh18_full[bh18_full['col_6'] == artikel_ref][['col_0', 'col_4', 'col_5', 'col_8', 'col_9', 'col_14', 'col_1']]
rows = rows.sort_values('col_1') # Sorteer op datum (col_1) om de volgorde te zien

print(f"Productiehistorie voor artikel {artikel_ref}:")
for i, row in rows.iterrows():
    print(f"Order: {row['col_0']} | Datum: {row['col_1']} | Status: {row['col_5']} | Plan: {row['col_8']} | Gemaakt: {row['col_9']}")

# Het idee is: als order N20024781 klaar is (Gereed), 
# moet het systeem automatisch N20024782 als 'Actieve' volgende order voorstellen.
# En als er in N20024782 méér wordt gemaakt dan de 5 geplande, moet dit misschien "doortellen" naar N20024783?

# Bekijk de som van gepland vs gemaakt voor dit artikel
totaal_gepland = rows['col_8'].sum()
totaal_gemaakt = rows['col_9'].sum()
print(f"\nTotaal voor dit artikel -> Gepland: {totaal_gepland}, Gemaakt: {totaal_gemaakt}")

Productiehistorie voor artikel EL4MEMS0FR0A10BCCBB0:
Order: N20024781 | Datum: 2026-04-24 06:15:23 | Status: Gereed | Plan: 17 | Gemaakt: 17
Order: N20024782 | Datum: 2026-05-01 06:15:23 | Status: Actief | Plan: 5 | Gemaakt: 10
Order: N20024783 | Datum: 2026-05-18 06:15:23 | Status: Actief | Plan: 2 | Gemaakt: 4
Order: N20025081 | Datum: 2026-06-01 00:06:16 | Status: Vrijgegeven | Plan: 0 | Gemaakt: 10

Totaal voor dit artikel -> Gepland: 24, Gemaakt: 41
